## ACS Data for ML

### "Adult"
 One of the most used default dataset is a 1994 ACS dataset use to predict whether an adult earned more than $50k per year

 [Adult ACS  Dataset on UCI Repository ](https://archive.ics.uci.edu/dataset/2/adult)


# Adult Income Classification with Logistic Regression and Model Comparison

This notebook walks through a common tabular machine learning example using the UCI Adult dataset. The goal is to predict whether a person earns more than `$50K` per year based on demographic and work-related variables such as age, education, occupation, and hours worked per week.

The dataset is widely used for teaching classification because it contains a mix of numeric and categorical variables, missing values, and a binary target. Historically, it comes from 1994 U.S. Census data and is hosted by the UCI Machine Learning Repository.

The notebook has two main parts:

1. Build and interpret a logistic regression model.
2. Compare logistic regression with several other classifiers.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## 1. Import the core libraries

These imports set up the tools used throughout the notebook:

- `pandas` and `numpy` for working with tabular data.
- `matplotlib` and `seaborn` for visualization.
- `scikit-learn` for splitting data, preprocessing features, building models, and evaluating predictions.

The first model used is logistic regression, which is a standard baseline for binary classification.


In [ ]:
try: from ucimlrepo import fetch_ucirepo 
except ImportError: 
    !pip install ucimlrepo
    from ucimlrepo import fetch_ucirepo


## 2. Load the dataset from the UCI repository

This section uses the `ucimlrepo` package to fetch the Adult dataset directly from UCI. In addition to the feature matrix and target column, the notebook prints metadata and variable descriptions so you can see what each column represents.

This is useful when teaching because it ties the modeling steps back to the source documentation.


In [ ]:
# load adult dataset
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.targets 
  
# metadata 
print(adult.metadata) 
  
# variable information 
print(adult.variables) 


## 3. Inspect the raw features

Before preprocessing, it is helpful to look at the first few rows. This gives a quick sense of the feature types and shows that several columns are categorical rather than purely numeric.


In [ ]:
#inspect the data for features

adult.data.features.head()

## 4. Load the raw CSV manually as an alternative

This notebook also demonstrates a second way to access the same dataset by reading the original CSV file from UCI directly. Doing this makes the preprocessing steps more explicit, because we manually assign column names and handle the raw text values ourselves.


In [ ]:
#or alternatively
# url for adult dataset
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
# column names for the dataset
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]
adultuci= pd.read_csv(url, names=column_names, sep=', ', engine='python')
adultuci.head()

## 5. Handle missing values

In the Adult dataset, missing categorical entries are represented by the string `?`. This block replaces those placeholders with `NaN` so pandas can treat them as missing values.

The notebook then drops rows with missing values and reports the dataset shape before and after cleaning. This is a simple strategy that keeps the workflow easy to follow, although in other projects you might prefer imputation instead of dropping data.


### Aside: strategies for handling missing values

This notebook drops rows with missing values, which is the simplest approach. `SimpleImputer` from `sklearn` is a common alternative that fills in missing values rather than discarding rows. Available strategies:

- `mean` — replace with the column mean (numeric only)
- `median` — replace with the column median; more robust to outliers than mean (numeric only)
- `most_frequent` — replace with the mode; works for both numeric and categorical columns
- `constant` — replace with a fixed value you specify via `fill_value`; works for both types

In a production pipeline, imputation is usually preferred because dropping rows can remove useful data and introduce bias if missingness is not random.

In [ ]:

# Replace '?' with NaN
adultuci.replace('?', np.nan, inplace=True)

# Drop rows with missing values
print("Shape before drop ",  adultuci.shape)
adultuci.dropna(inplace=True)
print("Shape after drop ",  adultuci.shape)


## 6. Convert categorical variables into numeric features

Most machine learning models in `scikit-learn` require numeric input. Since many Adult dataset columns are categorical, the notebook uses one-hot encoding with `pd.get_dummies()`.

With `drop_first=True`, one category from each encoded variable is omitted to reduce redundancy. The result is a fully numeric feature table that can be used by the classifiers.

> **Note on best practice:** Here encoding is applied to the full dataset before the train/test split. Strictly speaking, encoding should be fit only on training data and then applied to the test set to avoid any potential leakage of category membership. For this dataset the categories are fully represented in the training fold so the result is the same, but in production code you would handle this inside a pipeline using `sklearn.preprocessing.OneHotEncoder`.

In [ ]:
#one hot encoding for categorical variables
adultuci = pd.get_dummies(adultuci, drop_first=True)
adultuci.head()

## 7. Separate predictors and target

The target variable is whether income is greater than `$50K`. After encoding, that target appears as the binary column `income_>50K`.

This step assigns:

- `y` as the target to predict.
- `X` as the set of input features used to make that prediction.


## Aside on mising values   
`Simple Imputer` from `sklearn`
other options are: 

mean: Replaces missing values using the mean of the column. This strategy is only applicable to numerical data.

median: Replaces missing values using the median of the column. This strategy can be more robust than the mean, as it is less affected by outliers and is applicable to numerical data.

most_frequent: Replaces missing values using the mode (the most frequent value) of the column. This strategy can be used with both numerical and categorical (including string or object) data.

constant: Replaces missing values with a constant value that you specify through the fill_value parameter. This strategy can be used with both numerical and categorical data.

## 8. Split the data into training and test sets

To evaluate the model fairly, the data is split into:

- a training set used to fit the model
- a test set used only for evaluation

Using a held-out test set helps estimate how well the model generalizes to unseen examples.


In [ ]:
# split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train

## 9. Train a logistic regression pipeline

This block creates a modeling pipeline with two steps:

1. `StandardScaler()` standardizes the input features.
2. `LogisticRegression()` fits a binary classification model.

Using a pipeline keeps preprocessing and modeling together so the same transformations are applied consistently during training and prediction.


In [ ]:
# create a logistic regression model
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

yhat = model.predict(X_test)

## 10. Build a confusion matrix

After predicting on the test set, the notebook computes a confusion matrix. This helps break the results into four categories:

- true negatives
- false positives
- false negatives
- true positives

For a binary classification problem like income prediction, this gives more insight than accuracy alone.


In [ ]:
# calculate confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, yhat)
cm

In [ ]:
cm_df = pd.DataFrame(cm, 
                     index=["Actual Negative", "Actual Positive"], 
                     columns=["Predicted Negative", "Predicted Positive"])

print(cm_df)

## 11. Visualize model performance

The confusion matrix is displayed as a heatmap so it is easier to interpret. Large values on the diagonal indicate correct predictions, while off-diagonal values represent mistakes.


In [ ]:
plt.figure(figsize=(7, 7))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", annot_kws={"size": 16})
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

## 12. Calculate standard classification metrics

This section reports several common evaluation metrics:

- `Accuracy`: overall fraction of correct predictions.
- `Precision`: among predicted high-income cases, how many were actually high-income.
- `Recall`: among actual high-income cases, how many the model found.
- `F1 Score`: a balance between precision and recall.

Together, these metrics give a more complete picture of performance than any single number.

> **Class imbalance note:** About 76% of records in this dataset are labeled `<=50K`. This means a model that predicts `<=50K` for every example would achieve roughly 76% accuracy without learning anything. For imbalanced problems like this, Precision, Recall, F1, and ROC AUC are more informative than accuracy alone.

In [ ]:
# calculate accuracy
from sklearn.metrics import accuracy_score
print ("Accuracy Score", accuracy_score(y_test, yhat))
# calculate precision
from sklearn.metrics import precision_score
print ("Precision Score", precision_score(y_test, yhat))
# calculate recall
from sklearn.metrics import recall_score
print ("Recall Score", recall_score(y_test, yhat))
# calculate F1 score
from sklearn.metrics import f1_score
print ("F1 Score", f1_score(y_test, yhat))


## 13. Interpret logistic regression coefficients

One advantage of logistic regression is that it is relatively interpretable. Each coefficient shows how a feature is associated with the model's tendency to predict income above `$50K`, holding other encoded features fixed.

The table includes:

- `Coefficient`: the raw log-odds weight from the model.
- `Standardized Coefficient`: the raw coefficient scaled by the feature's standard deviation. This puts features on a comparable scale and helps rank their practical influence — a large standardized coefficient means that feature accounts for more variation in predictions.
- `Absolute Coefficient`: the magnitude of the raw coefficient, used for sorting.

Sorting by absolute size helps identify which encoded features have the strongest influence in the model.

In [ ]:
coefficients = model.named_steps['logisticregression'].coef_[0]

coefficients_df = pd.DataFrame({
    'Variable': X.columns,
    'Coefficient': coefficients,
    'Standardized Coefficient': np.std(X_train, 0) * coefficients,
    'Absolute Coefficient': np.abs(coefficients)
})
coefficients_df = coefficients_df.sort_values('Absolute Coefficient', ascending=False)
coefficients_df

## part 2 - Lets work on comparing different classifiers

## 14. Compare logistic regression with other classifiers

The second half of the notebook broadens the analysis by evaluating several different classification models on the same train/test split.

The common evaluation tools imported here will be reused for each model so the results are comparable.


In [ ]:
#Model Analytics 
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.metrics import roc_auc_score


### Decision Tree

A decision tree creates a sequence of if/then splits in the feature space. Trees are easy to visualize conceptually and can capture nonlinear relationships, but they can also overfit if left unconstrained.

> **Scaling note:** `StandardScaler` has no effect on tree-based models. Trees make decisions based on split thresholds, so rescaling features does not change any splits or predictions. It is kept in the pipeline here only to maintain a consistent structure across all models in this notebook.

In [ ]:
# Decision Tree Classifier
from sklearn.tree import DecisionTreeClassifier


# Define pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', DecisionTreeClassifier())
])




In [ ]:
# Fit and predict
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]


In [ ]:
# Evaluation metrics
print("=== Decision Tree ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

# Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Decision Tree - Confusion Matrix")
plt.show()

# ROC Curve
RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Decision Tree - ROC Curve")
plt.show()

### Random Forest

A random forest combines many decision trees and averages their predictions. This usually improves stability and predictive performance compared with a single tree by reducing variance.

> **Scaling note:** Like the single decision tree, random forests are invariant to feature scaling. The scaler in this pipeline has no practical effect here.

In [ ]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Random Forest - Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Random Forest - ROC Curve")
plt.show()

### Support Vector Machine

The support vector machine here uses an RBF kernel, which allows it to model nonlinear decision boundaries. SVMs can perform well on structured datasets, though they may be harder to interpret than logistic regression or trees.


In [ ]:
# SVM Classifier
from sklearn.svm import SVC
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(probability=True, kernel='rbf', C=1.0, random_state=42))
])


pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("SVM - Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("SVM - ROC Curve")
plt.show()

### Gradient Boosting

Gradient boosting builds a sequence of weak learners, where each new model focuses on correcting previous errors. It is often a strong performer on tabular datasets and can capture complex patterns.

> **Scaling note:** Gradient boosted trees are also invariant to monotonic feature transformations, so the scaler does not affect this model's predictions.

In [ ]:
# Gradient Boosting Classifier
from sklearn.ensemble import GradientBoostingClassifier
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== Gradient Boosting ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Gradient Boosting - Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Gradient Boosting - ROC Curve")
plt.show()

### Naive Bayes

Gaussian Naive Bayes is a simple probabilistic classifier that assumes features are conditionally independent given the class label. That assumption is usually unrealistic, but the method is fast and provides a useful baseline.

> **Scaling note:** Gaussian NB estimates the mean and variance of each feature internally, so `StandardScaler` does not change its predictions either.

In [ ]:
# Naive Bayes Classifier
from sklearn.naive_bayes import GaussianNB
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', GaussianNB())
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("=== Naive Bayes ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Naive Bayes - Confusion Matrix")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title("Naive Bayes - ROC Curve")
plt.show()

## 15. Compare ROC curves across all models

The final plot overlays ROC curves for several classifiers on the same axes. This makes it easier to compare their discrimination ability across classification thresholds rather than relying on a single cutoff.

A curve closer to the upper-left corner generally indicates better performance, and the diagonal reference line shows the behavior of random guessing.


In [ ]:
# Compare ROC Curves for all models
from sklearn.metrics import RocCurveDisplay

models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, kernel='rbf', C=1.0, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    'Naive Bayes': GaussianNB()
}


plt.figure(figsize=(10, 8))

for name, model in models.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', model)
    ])
    pipe.fit(X_train, y_train)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_proba, name=name, ax=plt.gca())

plt.title("Comparison of ROC Curves for Multiple Models")
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

## 16. Conclusion

### How did the models compare?

On the Adult dataset, the six classifiers generally fall into two tiers when evaluated by ROC AUC:

**Stronger performers**
- **Gradient Boosting** typically leads the group. It builds models sequentially, correcting prior errors at each step, which allows it to capture complex interactions in the data. Expect AUC around 0.93.
- **Random Forest** is close behind. Averaging many trees reduces variance and produces stable, well-calibrated probabilities. Expect AUC around 0.92.
- **SVM (RBF kernel)** also tends to perform well here because the kernel allows nonlinear boundaries, and the data separates reasonably well after scaling.

**Moderate performers**
- **Logistic Regression** is a strong baseline given its simplicity. It performs competitively on this dataset because many of the income-related signals (education, hours worked, occupation) have roughly linear relationships with log-odds of high income. Expect AUC around 0.89–0.90.
- **Decision Tree** (unpruned) often matches or slightly exceeds logistic regression on accuracy but can overfit, which shows up as a lower or noisier ROC curve compared to the ensemble methods.

**Weakest baseline**
- **Naive Bayes** typically scores lowest here. Its core assumption — that all features are independent of one another given the class label — is clearly violated in this dataset (education level and occupation are correlated, for example). It remains useful as a sanity-check baseline.

### Key takeaways

1. **Accuracy alone is misleading.** Because ~76% of rows are labeled `<=50K`, a trivial model that always predicts the majority class would score 76% accuracy. ROC AUC and F1 are better signals of real discriminative ability.

2. **Ensemble methods outperform single models.** Random Forest and Gradient Boosting both improve on the single Decision Tree by reducing variance (bagging) or bias (boosting). This is a general pattern on tabular data.

3. **Logistic regression is a strong, interpretable baseline.** It is worth fitting first before reaching for more complex models. Its coefficients can be used to understand which features drive predictions, as shown in section 13.

4. **Model complexity has costs.** Gradient Boosting and SVM are slower to train and harder to interpret. For a production setting, the choice between a logistic regression and a gradient boosting model is often a tradeoff between explainability, latency, and predictive performance rather than a pure accuracy contest.

5. **Dataset context matters.** This dataset is from 1994 U.S. Census data. The income threshold of $50K is not adjusted for inflation, and the feature set encodes historical demographic patterns. Any model trained here should not be applied to real income prediction decisions without careful review of its fairness properties.